# Session 2 · Prepare data (Colab)

Turns FineWeb-Edu sample-10BT (14 files, 28.5 GB) into about 10B tokens of shards on your Google Drive, plus a report.

- **Runtime:** any; the work is CPU-heavy, so more CPU cores means faster. An L4 or A100 runtime has more cores than the basic CPU one.
- **Drive space:** about 20 GB of shards; cell 2 checks this.
- **Resumable:** if Colab disconnects, reopen this notebook and *Run all*. Finished files are skipped.
- Keep the tab open (or use Colab Pro+ background execution). It takes a few hours.

In [ ]:
# 1. Keys, Drive, latest code, packages
import os, subprocess
from google.colab import userdata, drive
for k in ['HF_TOKEN','WANDB_API_KEY','GIT_NAME','GIT_EMAIL','GITHUB_TOKEN']:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass
drive.mount('/content/drive')
os.environ['CHOTU_STORAGE'] = '/content/drive/MyDrive/chotu'
os.makedirs(os.environ['CHOTU_STORAGE'], exist_ok=True)
tok = os.environ.get('GITHUB_TOKEN')
url = f'https://{tok}@github.com/aditya950820/Chotu.git' if tok else 'https://github.com/aditya950820/Chotu.git'
if not os.path.exists('/content/Chotu'): subprocess.run(['git','clone',url,'/content/Chotu'],check=True)
%cd /content/Chotu
!git pull --rebase -q && git config user.name "$GIT_NAME" && git config user.email "$GIT_EMAIL"
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!nproc

In [ ]:
# 2. Drive space check (need ~22 GB free for shards)
import shutil
free = shutil.disk_usage(os.environ['CHOTU_STORAGE']).free/1e9
print(f'Drive free: {free:.1f} GB'); assert free > 22, 'Not enough Drive space: upgrade Google One storage or free up space.'

In [ ]:
# 3. Quick test (~1 min): tokenizer checks + 2 small shards in chotu/test/
!python scripts/prepare_data.py --max-docs 2000 --max-files 2

In [ ]:
# 4. Full run (a few hours, resumable)
!python scripts/prepare_data.py

In [ ]:
# 5. Read the report: token count, tokenizer checks, 10 samples
from IPython.display import Markdown
Markdown(open(os.environ['CHOTU_STORAGE'] + '/reports/data_report.md').read())

## Gate
- About 10B train tokens, and a validation split exists
- You read the 10 samples: clean English, documents separated, no markup junk

Then add your entry to `docs/PROGRESS.md` (numbers from the report) and push.

In [ ]:
!git add docs/ && git commit -m "Session 2: data prepared" && git pull --rebase -q && git push